In [ ]:
# 日频反转拆为排序、队列建仓、日度更新和月度复合。
def get_reversal_legs(weight, signal, size_neutral=False):
    valid = np.isfinite(weight) & (weight > 0) & np.isfinite(signal)
    selected = np.flatnonzero(valid)
    long_members = np.zeros(len(weight), dtype=bool)
    short_members = np.zeros(len(weight), dtype=bool)
    if len(selected) < 100:
        return long_members, short_members
    if size_neutral:
        size_rank = pd.Series(weight[selected]).rank(method='first')
        size_group = pd.qcut(size_rank, q=10, labels=False).to_numpy()
        for group in range(10):
            members = selected[size_group == group]
            if len(members) < 10:
                continue
            signal_rank = pd.Series(signal[members]).rank(method='first')
            signal_group = pd.qcut(signal_rank, q=10, labels=False).to_numpy()
            long_members[members[signal_group == 0]] = True
            short_members[members[signal_group == 9]] = True
    else:
        signal_rank = pd.Series(signal[selected]).rank(method='first')
        signal_group = pd.qcut(signal_rank, q=10, labels=False).to_numpy()
        long_members[selected[signal_group == 0]] = True
        short_members[selected[signal_group == 9]] = True
    return long_members, short_members


In [ ]:
def make_reversal_cohort(weight, long_members, short_members):
    long_weight = np.where(long_members, weight, 0.0)
    short_weight = np.where(short_members, weight, 0.0)
    if long_weight.sum() == 0 or short_weight.sum() == 0:
        return None
    return [long_weight / long_weight.sum(), short_weight / short_weight.sum()]


In [ ]:
def update_reversal_cohort(cohort, today):
    long_weight, short_weight = cohort
    held_members = (long_weight + short_weight) > 0
    missing_held = int((held_members & ~np.isfinite(today)).sum())
    gross_return = 1 + np.nan_to_num(today, nan=0.0)
    long_return = np.sum(long_weight * (gross_return - 1))
    short_return = np.sum(short_weight * (gross_return - 1))
    cohort[0] = long_weight * gross_return / (1 + long_return)
    cohort[1] = short_weight * gross_return / (1 + short_return)
    return long_return - short_return, missing_held


In [ ]:
def compound_reversal_months(daily):
    monthly_returns = []
    months = []
    for month, month_returns in daily.groupby(daily.index.to_period('M')):
        observed_returns = month_returns.dropna()
        if len(observed_returns) >= 15:
            monthly_returns.append((1 + observed_returns).prod() - 1)
        else:
            monthly_returns.append(np.nan)
        months.append(month)
    return pd.Series(
        monthly_returns,
        index=pd.PeriodIndex(months, freq='M'),
        name='reversal',
    )


In [ ]:
def get_daily_reversal(df, monthly_final, size_neutral=False):
    returns = df.pivot(index=date, columns=stock_code, values=daily_ret)
    returns = returns.sort_index()
    signals = df.pivot(index=date, columns=stock_code, values=daily_reversal)
    signals = signals.reindex_like(returns).shift(1)
    caps = monthly_final.pivot(
        index=ym, columns=stock_code, values=monthly_market_value
    )
    caps = caps.reindex(columns=returns.columns)
    cohorts = [None] * 5
    daily_spread = []
    missing_held = 0

    for position, day in enumerate(returns.index):
        month = day.to_period('M')
        if month not in caps.index:
            cohorts = [None] * 5
            daily_spread.append(np.nan)
            continue
        weight = caps.loc[month].to_numpy(dtype=float)
        signal = signals.iloc[position].to_numpy(dtype=float)
        today = returns.iloc[position].to_numpy(dtype=float)
        long_members, short_members = get_reversal_legs(
            weight, signal, size_neutral
        )
        cohorts[position % 5] = make_reversal_cohort(
            weight, long_members, short_members
        )
        cohort_returns = []
        for cohort in cohorts:
            if cohort is None:
                continue
            cohort_return, missing = update_reversal_cohort(cohort, today)
            cohort_returns.append(cohort_return)
            missing_held += missing
        if len(cohort_returns) == 5:
            daily_spread.append(np.mean(cohort_returns))
        else:
            daily_spread.append(np.nan)

    daily = pd.Series(daily_spread, index=returns.index, name='reversal')
    monthly_reversal = compound_reversal_months(daily)
    portfolio_type = 'size-neutral' if size_neutral else 'unconditional'
    print('reversal {}: missing held-day observations {}'.format(
        portfolio_type, missing_held
    ))
    portfolio_months = pd.PeriodIndex(
        monthly_final[ym].unique(), freq='M'
    ).sort_values()
    return monthly_reversal.reindex(portfolio_months)


In [ ]:
anomaly_uncon['reversal'] = get_daily_reversal(df, monthly_final, False)


In [ ]:
anomaly_neutral['reversal'] = get_daily_reversal(df, monthly_final, True)


In [ ]:
print('查看全部异象每列的实际有效月数')


In [ ]:
reversal_coverage = pd.DataFrame(
    {
        'unconditional': anomaly_uncon.count(),
        'size_neutral': anomaly_neutral.count(),
    }
)


In [ ]:
display(reversal_coverage)


# 记录本地数据口径与覆盖限制，供 src_jup 的模型回归解释。

